# NTO AI Final — Advanced GPU Pipeline

Новый ноутбук для Kaggle GPU:
- генерация кандидатов (ALS U2I + ALS I2I + author + genre + global),
- расширенные ALS-агрегаты по 10 ближайшим соседям из истории пользователя,
- CatBoostRanker (GPU),
- локальная валидация NDCG@20,
- строгая валидация submission-контракта.


In [ ]:
# CELL 1 — Imports / config
import os
import gc
import math
import csv
import warnings
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import pandas as pd
from pandas.errors import ParserError
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostRanker, Pool
from implicit.als import AlternatingLeastSquares

warnings.filterwarnings("ignore")

SEED = 42
TOP_K = 20
CANDS_PER_USER = 260
VALID_DAYS = 31
INCIDENT_DATE = "2025-10-01"
ALS_FACTORS = 128
ALS_ITERS = 24
ALS_REG = 0.02
CHUNK_USERS = 350

DATA_DIRS = [
    "/kaggle/input/nto-ai",
    "/kaggle/input/datasets/artemnazemtsev/nto-ai",
    "./data"
]


def detect_data_dir(paths):
    for p in paths:
        if os.path.exists(os.path.join(p, "interactions.csv")):
            return p
    raise FileNotFoundError("Cannot find interactions.csv")


def detect_sep(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        sample = f.read(8192)
    try:
        return csv.Sniffer().sniff(sample, delimiters=[",", ";", "	", "|"]).delimiter
    except Exception:
        return ","


def read_csv_safe(path: str, parse_dates=None):
    sep = detect_sep(path)

    # Fast path: C engine
    try:
        return pd.read_csv(path, sep=sep, parse_dates=parse_dates, low_memory=False)
    except (ParserError, pd.errors.ParserError) as e:
        print(f"[WARN] ParserError for {os.path.basename(path)} with C engine: {e}")

    # Robust path: Python engine (note: low_memory is NOT supported here)
    try:
        print("[WARN] Fallback -> python engine + on_bad_lines='skip'")
        return pd.read_csv(
            path,
            sep=sep,
            parse_dates=parse_dates,
            engine="python",
            on_bad_lines="skip",
        )
    except ValueError as e:
        # Extra-safe retry if pandas version still complains about options
        print(f"[WARN] Python engine read failed: {e}")
        print("[WARN] Retry -> python engine + dtype=str")
        return pd.read_csv(
            path,
            sep=sep,
            parse_dates=parse_dates,
            engine="python",
            on_bad_lines="skip",
            dtype=str,
        )


def mem_usage_mb(df: pd.DataFrame) -> float:
    return float(df.memory_usage(deep=True).sum() / 1024**2)


def print_mem(df: pd.DataFrame, name: str):
    print(f"{name}: rows={len(df):,}, mem={mem_usage_mb(df):.2f} MB")




In [ ]:
# CELL 2 — Load data (robust typed + malformed CSV tolerant)
DATA_DIR = detect_data_dir(DATA_DIRS)
print("DATA_DIR:", DATA_DIR)

# 1) interactions
interactions = read_csv_safe(
    os.path.join(DATA_DIR, "interactions.csv"),
    parse_dates=["event_ts"],
)

required_cols = ["user_id", "edition_id", "event_type", "event_ts"]
missing = [c for c in required_cols if c not in interactions.columns]
if missing:
    raise ValueError(f"interactions.csv missing required columns: {missing}")

for c in ["user_id", "edition_id", "event_type"]:
    interactions[c] = pd.to_numeric(interactions[c], errors="coerce")

if "rating" not in interactions.columns:
    interactions["rating"] = np.nan
interactions["rating"] = pd.to_numeric(interactions["rating"], errors="coerce").astype("float32")

before_rows = len(interactions)
interactions = interactions.dropna(subset=["user_id", "edition_id", "event_type", "event_ts"]).copy()
after_rows = len(interactions)
if after_rows < before_rows:
    print(f"Dropped invalid interaction rows: {before_rows - after_rows:,}")

interactions["user_id"] = interactions["user_id"].astype("int64")
interactions["edition_id"] = interactions["edition_id"].astype("int64")
interactions["event_type"] = interactions["event_type"].astype("int8")

# 2) targets
targets = read_csv_safe(os.path.join(DATA_DIR, "targets.csv"))
if "user_id" not in targets.columns:
    raise ValueError("targets.csv missing required column user_id")
targets["user_id"] = pd.to_numeric(targets["user_id"], errors="coerce")
targets = targets.dropna(subset=["user_id"]).copy()
targets["user_id"] = targets["user_id"].astype("int64")

# 3) справочники
editions = read_csv_safe(os.path.join(DATA_DIR, "editions.csv"))
book_genres = read_csv_safe(os.path.join(DATA_DIR, "book_genres.csv"))

users_path = os.path.join(DATA_DIR, "users.csv")
users = read_csv_safe(users_path) if os.path.exists(users_path) else pd.DataFrame({"user_id": interactions["user_id"].unique()})

if "user_id" not in users.columns:
    users = pd.DataFrame({"user_id": interactions["user_id"].unique()})
users["user_id"] = pd.to_numeric(users["user_id"], errors="coerce")
users = users.dropna(subset=["user_id"]).copy()
users["user_id"] = users["user_id"].astype("int64")

for col in ["edition_id", "book_id", "author_id", "language_id", "publisher_id", "publication_year", "age_restriction"]:
    if col in editions.columns:
        editions[col] = pd.to_numeric(editions[col], errors="coerce")

for col in ["book_id", "genre_id"]:
    if col in book_genres.columns:
        book_genres[col] = pd.to_numeric(book_genres[col], errors="coerce")

if "edition_id" in editions.columns:
    e_before = len(editions)
    editions = editions.dropna(subset=["edition_id"]).copy()
    editions["edition_id"] = editions["edition_id"].astype("int64")
    if len(editions) < e_before:
        print(f"Dropped invalid editions rows: {e_before - len(editions):,}")

if set(["book_id", "genre_id"]).issubset(book_genres.columns):
    bg_before = len(book_genres)
    book_genres = book_genres.dropna(subset=["book_id", "genre_id"]).copy()
    book_genres["book_id"] = book_genres["book_id"].astype("int64")
    book_genres["genre_id"] = book_genres["genre_id"].astype("int64")
    if len(book_genres) < bg_before:
        print(f"Dropped invalid book_genres rows: {bg_before - len(book_genres):,}")

print_mem(interactions, "interactions")
print_mem(targets, "targets")
print_mem(editions, "editions")
print_mem(book_genres, "book_genres")
print_mem(users, "users")



In [ ]:
# CELL 3 — Split + helper metrics
POS_EVENTS = [1, 2]

T_MAX = interactions["event_ts"].max()
T_INCIDENT = pd.Timestamp(INCIDENT_DATE)
if T_INCIDENT < interactions["event_ts"].min() or T_INCIDENT > T_MAX:
    T_INCIDENT = T_MAX - pd.Timedelta(days=VALID_DAYS)
T_VALID_START = T_INCIDENT - pd.Timedelta(days=VALID_DAYS)

train_log = interactions[interactions["event_ts"] < T_VALID_START].copy()
valid_log = interactions[(interactions["event_ts"] >= T_VALID_START) & (interactions["event_ts"] < T_INCIDENT)].copy()

print("Train range:", train_log["event_ts"].min(), "->", train_log["event_ts"].max())
print("Valid range:", valid_log["event_ts"].min(), "->", valid_log["event_ts"].max())
print("train rows:", len(train_log), "valid rows:", len(valid_log))


def ndcg_at_k_binary(relevance, k=20):
    rel = np.asarray(relevance, dtype=np.float32)[:k]
    if rel.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, rel.size + 2, dtype=np.float32))
    dcg = float((rel * discounts).sum())
    ideal_count = int(rel.sum())
    if ideal_count == 0:
        return 0.0
    ideal_rel = np.concatenate([np.ones(min(k, ideal_count), dtype=np.float32), np.zeros(max(0, k - ideal_count), dtype=np.float32)])
    idcg = float((ideal_rel * (1.0 / np.log2(np.arange(2, ideal_rel.size + 2, dtype=np.float32)))).sum())
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_ndcg20(pred_df, gt_pairs, k=20):
    # pred_df: user_id, edition_id, rank
    gt_map = defaultdict(set)
    for u,e in gt_pairs:
        gt_map[u].add(e)

    vals = []
    for uid, grp in pred_df.sort_values(["user_id", "rank"]).groupby("user_id"):
        rel = [1.0 if eid in gt_map.get(uid, set()) else 0.0 for eid in grp["edition_id"].tolist()[:k]]
        vals.append(ndcg_at_k_binary(rel, k=k))
    return float(np.mean(vals)) if vals else 0.0


In [ ]:
# CELL 4 — Base feature builders

def build_user_item_features(log_df, editions_df, users_df, book_genres_df, ref_time):
    pos = log_df[log_df["event_type"].isin(POS_EVENTS)].copy()

    # item counters
    i_pop = pos.groupby("edition_id").size().astype("float32").rename("edition_pop")
    i_read = pos[pos["event_type"].eq(2)].groupby("edition_id").size().astype("float32").rename("read_pop")
    i_wish = pos[pos["event_type"].eq(1)].groupby("edition_id").size().astype("float32").rename("wish_pop")
    i_u = pos.groupby("edition_id")["user_id"].nunique().astype("float32").rename("edition_unique_users")
    i_recent = pos[pos["event_ts"] >= (ref_time - pd.Timedelta(days=31))].groupby("edition_id").size().astype("float32").rename("edition_recent_pop")

    item = editions_df[[c for c in ["edition_id","book_id","author_id","publication_year","language_id","publisher_id","age_restriction"] if c in editions_df.columns]].copy()
    for s in [i_pop, i_read, i_wish, i_u, i_recent]:
        item = item.merge(s.reset_index(), on="edition_id", how="left")
    gcount = book_genres_df.groupby("book_id")["genre_id"].nunique().astype("float32").rename("book_genre_count")
    if "book_id" in item.columns:
        item = item.merge(gcount.reset_index(), on="book_id", how="left")

    for c in ["edition_pop","read_pop","wish_pop","edition_unique_users","edition_recent_pop","book_genre_count"]:
        if c in item.columns:
            item[c] = item[c].fillna(0).astype("float32")
    item["pop_log"] = np.log1p(item.get("edition_pop", 0)).astype("float32")
    item["recent_pop_log"] = np.log1p(item.get("edition_recent_pop", 0)).astype("float32")
    item["read_ratio"] = (item.get("read_pop", 0) / (item.get("edition_pop", 0) + 1)).astype("float32")
    item["wish_ratio"] = (item.get("wish_pop", 0) / (item.get("edition_pop", 0) + 1)).astype("float32")

    # user counters
    universe_users = np.union1d(users_df["user_id"].unique(), targets["user_id"].unique())
    user = pd.DataFrame({"user_id": universe_users})
    u_act = pos.groupby("user_id").size().astype("float32").rename("user_activity")
    u_uniq = pos.groupby("user_id")["edition_id"].nunique().astype("float32").rename("user_uniq_editions")
    u_read = pos[pos["event_type"].eq(2)].groupby("user_id").size().astype("float32").rename("user_read_count")
    u_wish = pos[pos["event_type"].eq(1)].groupby("user_id").size().astype("float32").rename("user_wish_count")

    for s in [u_act, u_uniq, u_read, u_wish]:
        user = user.merge(s.reset_index(), on="user_id", how="left")

    if "gender" in users_df.columns:
        user = user.merge(users_df[["user_id", "gender"]], on="user_id", how="left")
    if "age" in users_df.columns:
        user = user.merge(users_df[["user_id", "age"]], on="user_id", how="left")

    for c in ["user_activity","user_uniq_editions","user_read_count","user_wish_count","gender","age"]:
        if c in user.columns:
            user[c] = pd.to_numeric(user[c], errors="coerce").fillna(0).astype("float32")
    user["user_activity_log"] = np.log1p(user.get("user_activity", 0)).astype("float32")
    user["user_read_ratio"] = (user.get("user_read_count", 0) / (user.get("user_activity", 0) + 1)).astype("float32")
    user["user_wish_ratio"] = (user.get("user_wish_count", 0) / (user.get("user_activity", 0) + 1)).astype("float32")

    return user, item


def build_profiles(log_df, editions_df, book_genres_df):
    pos = log_df[log_df["event_type"].isin(POS_EVENTS)].copy()
    e2b = editions_df[["edition_id", "book_id"]].drop_duplicates()
    e2a = editions_df[["edition_id", "author_id"]].drop_duplicates()

    ug = (pos.merge(e2b, on="edition_id", how="left")
            .merge(book_genres_df[["book_id", "genre_id"]], on="book_id", how="left")
            .dropna(subset=["genre_id"]))
    ug = ug.groupby(["user_id", "genre_id"]).size().rename("cnt").reset_index()
    ug["user_genre_affinity"] = (ug["cnt"] / ug.groupby("user_id")["cnt"].transform("sum")).astype("float32")

    ua = (pos.merge(e2a, on="edition_id", how="left")
            .dropna(subset=["author_id"]))
    ua = ua.groupby(["user_id", "author_id"]).size().rename("cnt").reset_index()
    ua["user_author_affinity"] = (ua["cnt"] / ua.groupby("user_id")["cnt"].transform("sum")).astype("float32")

    return ug[["user_id", "genre_id", "user_genre_affinity"]], ua[["user_id", "author_id", "user_author_affinity"]]


In [ ]:
# CELL 5 — ALS fit + candidate generation

def fit_als(log_df):
    pos = log_df[log_df["event_type"].isin(POS_EVENTS)].copy()
    pos["base_w"] = pos["event_type"].map({1: 1.0, 2: 2.0}).astype("float32")
    mx = pos["event_ts"].max()
    pos["days_old"] = (mx - pos["event_ts"]).dt.days.clip(lower=0)
    pos["w"] = (pos["base_w"] * np.exp(-np.log(2) * pos["days_old"] / 30)).astype("float32")

    u_enc = LabelEncoder(); i_enc = LabelEncoder()
    u_idx = u_enc.fit_transform(pos["user_id"])
    i_idx = i_enc.fit_transform(pos["edition_id"])
    mat = csr_matrix((pos["w"].to_numpy(), (u_idx, i_idx)), shape=(len(u_enc.classes_), len(i_enc.classes_)))

    als = AlternatingLeastSquares(
        factors=ALS_FACTORS,
        iterations=ALS_ITERS,
        regularization=ALS_REG,
        random_state=SEED,
        use_gpu=True,
    )
    als.fit(mat)

    model = {
        "als": als,
        "X": mat,
        "u2als": {u:i for i,u in enumerate(u_enc.classes_)},
        "e2als": {e:i for i,e in enumerate(i_enc.classes_)},
        "als_items": i_enc.classes_.astype("int64"),
        "seen_pairs": set(zip(pos["user_id"].to_numpy(), pos["edition_id"].to_numpy())),
        "user_hist": pos.sort_values("event_ts").groupby("user_id")["edition_id"].apply(lambda x: list(x)[-8:]).to_dict(),
    }
    return model


def generate_candidates(user_ids, item_feat, ug, ua, als_bundle):
    als = als_bundle["als"]
    X = als_bundle["X"]
    u2als = als_bundle["u2als"]
    e2als = als_bundle["e2als"]
    als_items = als_bundle["als_items"]
    seen_pairs = als_bundle["seen_pairs"]
    user_hist = als_bundle["user_hist"]

    global_top = item_feat.sort_values("pop_log", ascending=False)["edition_id"].head(300).tolist()
    gidx = (item_feat[["edition_id", "book_id", "pop_log"]]
            .merge(book_genres[["book_id", "genre_id"]], on="book_id", how="left")
            .sort_values(["genre_id", "pop_log"], ascending=[True, False])
            .groupby("genre_id")["edition_id"].apply(lambda x: x.head(60).tolist()).to_dict())
    aidx = (item_feat.sort_values(["author_id", "pop_log"], ascending=[True, False])
            .groupby("author_id")["edition_id"].apply(lambda x: x.head(40).tolist()).to_dict())

    ug_map = ug.sort_values(["user_id", "user_genre_affinity"], ascending=[True, False]).groupby("user_id")["genre_id"].apply(list).to_dict()
    ua_map = ua.sort_values(["user_id", "user_author_affinity"], ascending=[True, False]).groupby("user_id")["author_id"].apply(list).to_dict()

    rows = []
    uids = list(user_ids)
    for st in range(0, len(uids), CHUNK_USERS):
        batch = uids[st:st+CHUNK_USERS]
        for uid in batch:
            cand = {}
            def add(eid, score, src_idx):
                if (uid, eid) in seen_pairs:
                    return
                if eid not in cand:
                    cand[eid] = [0.0, 0, 0, 0, 0, 0]
                cand[eid][0] += float(score)
                cand[eid][src_idx] = 1

            # ALS U2I
            if uid in u2als:
                rec_ids, rec_scores = als.recommend(u2als[uid], X[u2als[uid]], N=140, filter_already_liked_items=True)
                for r, idx in enumerate(rec_ids):
                    add(int(als_items[idx]), 4.0 / (r + 1), 4)

            # ALS I2I
            for heid in user_hist.get(uid, []):
                if heid not in e2als:
                    continue
                ids, sims = als.similar_items(e2als[heid], N=18)
                for r, (idx, sim) in enumerate(zip(ids, sims)):
                    eid = int(als_items[idx])
                    if eid != heid:
                        add(eid, 3.2 * float(sim) / (r + 1), 5)

            # genre / author
            for rg, gid in enumerate(ug_map.get(uid, [])[:12]):
                for eid in gidx.get(gid, []):
                    add(int(eid), 1.3 / (rg + 1), 1)
            for ra, aid in enumerate(ua_map.get(uid, [])[:12]):
                for eid in aidx.get(aid, []):
                    add(int(eid), 1.3 / (ra + 1), 2)

            # global fallback
            for rg, eid in enumerate(global_top):
                add(int(eid), 0.45 / (rg + 1), 3)

            top = sorted(cand.items(), key=lambda kv: kv[1][0], reverse=True)[:CANDS_PER_USER]
            for eid, info in top:
                rows.append((uid, eid, info[0], info[1], info[2], info[3], info[4], info[5]))

        gc.collect()

    cands = pd.DataFrame(rows, columns=["user_id","edition_id","cand_score","src_genre","src_author","src_global","src_als_u2i","src_als_i2i"])
    cands = cands.sort_values(["user_id", "cand_score"], ascending=[True, False]).drop_duplicates(["user_id","edition_id"], keep="first")
    cands["cand_score"] = cands["cand_score"].astype("float32")
    for c in ["src_genre","src_author","src_global","src_als_u2i","src_als_i2i"]:
        cands[c] = cands[c].astype("int8")
    return cands

train_user_features, train_item_features = build_user_item_features(train_log, editions, users, book_genres, T_INCIDENT)
train_ug, train_ua = build_profiles(train_log, editions, book_genres)
train_als_bundle = fit_als(train_log)
train_users = valid_log["user_id"].drop_duplicates().tolist()
train_cands = generate_candidates(train_users, train_item_features, train_ug, train_ua, train_als_bundle)
print_mem(train_cands, "train_cands")


In [ ]:
# CELL 6 — ALS 10-neighbor aggregate features (advanced)

def build_als_neighbor_features(cands_df, als_bundle, max_hist=10, knn=10):
    als = als_bundle["als"]
    e2als = als_bundle["e2als"]
    user_hist = als_bundle["user_hist"]
    als_items = als_bundle["als_items"]

    V = np.asarray(als.item_factors, dtype=np.float32)
    Vn = V / np.clip(np.linalg.norm(V, axis=1, keepdims=True), 1e-9, None)

    # optional rating dictionary from train log
    train_reads = train_log[(train_log["event_type"].eq(2)) & train_log["rating"].notna()][["edition_id", "rating"]]
    rating_map = train_reads.groupby("edition_id")["rating"].mean().astype("float32").to_dict()

    out = []
    for uid, grp in cands_df.groupby("user_id", sort=False):
        hist = [eid for eid in user_hist.get(uid, []) if eid in e2als][-max_hist:]
        if len(hist) == 0:
            z = np.zeros((len(grp), 10), dtype=np.float32)
            for i, (idx, row) in enumerate(grp.iterrows()):
                out.append((idx, *z[i]))
            continue

        hist_idx = np.array([e2als[e] for e in hist], dtype=np.int32)
        hist_vecs = Vn[hist_idx]

        cand_eids = grp["edition_id"].to_numpy()
        cand_idx = np.array([e2als.get(e, -1) for e in cand_eids], dtype=np.int32)

        for row_idx, ci in zip(grp.index.to_numpy(), cand_idx):
            if ci < 0:
                out.append((row_idx, 0,0,0,0,0,0,0,0,0,0))
                continue

            sims = (hist_vecs @ Vn[ci].reshape(-1,1)).ravel()
            if sims.size > knn:
                top_idx = np.argpartition(sims, -knn)[-knn:]
                top_sims = np.sort(sims[top_idx])[::-1]
                top_hist = hist_idx[top_idx]
            else:
                order = np.argsort(sims)[::-1]
                top_sims = sims[order]
                top_hist = hist_idx[order]

            # rating агрегации по соседям
            neigh_eids = als_items[top_hist]
            neigh_ratings = np.array([rating_map.get(int(eid), 0.0) for eid in neigh_eids], dtype=np.float32)
            ws = np.clip(top_sims, 0, None) + 1e-6

            mean_sim = float(np.mean(top_sims)) if top_sims.size else 0.0
            max_sim = float(np.max(top_sims)) if top_sims.size else 0.0
            min_sim = float(np.min(top_sims)) if top_sims.size else 0.0
            std_sim = float(np.std(top_sims)) if top_sims.size else 0.0
            q25 = float(np.quantile(top_sims, 0.25)) if top_sims.size else 0.0
            q50 = float(np.quantile(top_sims, 0.50)) if top_sims.size else 0.0
            q75 = float(np.quantile(top_sims, 0.75)) if top_sims.size else 0.0
            mean_rat = float(np.mean(neigh_ratings)) if neigh_ratings.size else 0.0
            w_rat = float(np.average(neigh_ratings, weights=ws)) if neigh_ratings.size else 0.0
            pos_ratio = float(np.mean(top_sims > 0.2)) if top_sims.size else 0.0

            out.append((row_idx, mean_sim, max_sim, min_sim, std_sim, q25, q50, q75, mean_rat, w_rat, pos_ratio))

    feat = pd.DataFrame(out, columns=[
        "idx", "als_nb_mean_sim", "als_nb_max_sim", "als_nb_min_sim", "als_nb_std_sim",
        "als_nb_q25_sim", "als_nb_q50_sim", "als_nb_q75_sim",
        "als_nb_mean_rating", "als_nb_weighted_rating", "als_nb_pos_ratio"
    ]).set_index("idx")

    for c in feat.columns:
        feat[c] = feat[c].astype("float32")
    return feat

train_nb = build_als_neighbor_features(train_cands, train_als_bundle, max_hist=10, knn=10)
train_cands = train_cands.join(train_nb, how="left").fillna(0)
print_mem(train_cands, "train_cands+als_nb")


In [ ]:
# CELL 7 — Build ranking train table + local validation
valid_pos_pairs = valid_log[valid_log["event_type"].isin(POS_EVENTS)][["user_id","edition_id"]].drop_duplicates()
valid_pos_pairs["target"] = 1

train_rank = train_cands.merge(valid_pos_pairs, on=["user_id","edition_id"], how="left")
train_rank["target"] = train_rank["target"].fillna(0).astype("int8")
train_rank = train_rank.merge(train_user_features, on="user_id", how="left")
train_rank = train_rank.merge(train_item_features, on="edition_id", how="left")

# compact affinity aggregates
ug_user = train_ug.groupby("user_id")["user_genre_affinity"].agg(["mean","max"]).reset_index().rename(columns={"mean":"u_genre_aff_mean","max":"u_genre_aff_max"})
ua_user = train_ua.groupby("user_id")["user_author_affinity"].agg(["mean","max"]).reset_index().rename(columns={"mean":"u_author_aff_mean","max":"u_author_aff_max"})
train_rank = train_rank.merge(ug_user, on="user_id", how="left").merge(ua_user, on="user_id", how="left")

for c in train_rank.columns:
    if str(train_rank[c].dtype) == "float64":
        train_rank[c] = train_rank[c].astype("float32")
train_rank = train_rank.fillna(0)
print_mem(train_rank, "train_rank")
print("pos_rate:", float(train_rank["target"].mean()))


In [ ]:
# CELL 8 — Train CatBoostRanker on GPU
feature_cols = [
    "cand_score", "src_genre", "src_author", "src_global", "src_als_u2i", "src_als_i2i",
    "als_nb_mean_sim", "als_nb_max_sim", "als_nb_min_sim", "als_nb_std_sim",
    "als_nb_q25_sim", "als_nb_q50_sim", "als_nb_q75_sim",
    "als_nb_mean_rating", "als_nb_weighted_rating", "als_nb_pos_ratio",
    "user_activity", "user_uniq_editions", "user_read_count", "user_wish_count", "user_activity_log", "user_read_ratio", "user_wish_ratio",
    "edition_pop", "edition_recent_pop", "edition_unique_users", "read_pop", "wish_pop",
    "pop_log", "recent_pop_log", "read_ratio", "wish_ratio", "book_genre_count",
    "publication_year", "language_id", "publisher_id", "author_id", "age_restriction",
    "gender", "age", "u_genre_aff_mean", "u_genre_aff_max", "u_author_aff_mean", "u_author_aff_max"
]
feature_cols = [c for c in feature_cols if c in train_rank.columns]
print("features:", len(feature_cols))

train_rank = train_rank.sort_values("user_id").reset_index(drop=True)
pool = Pool(
    data=train_rank[feature_cols],
    label=train_rank["target"],
    group_id=train_rank["user_id"],
)

ranker = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=20",
    iterations=1600,
    learning_rate=0.035,
    depth=8,
    l2_leaf_reg=8,
    random_seed=SEED,
    task_type="GPU",
    verbose=200,
)
ranker.fit(pool)

# local validation score
train_rank["pred"] = ranker.predict(Pool(train_rank[feature_cols]))
valid_pred = train_rank.sort_values(["user_id","pred","edition_id"], ascending=[True,False,True])[["user_id","edition_id","pred"]].copy()
valid_pred = valid_pred.drop_duplicates(["user_id","edition_id"], keep="first")
valid_pred["rank"] = valid_pred.groupby("user_id").cumcount() + 1
valid_pred = valid_pred[valid_pred["rank"] <= TOP_K][["user_id","edition_id","rank"]]

gt = set(map(tuple, valid_pos_pairs[["user_id","edition_id"]].to_numpy()))
print("Local NDCG@20:", round(evaluate_ndcg20(valid_pred, gt, k=TOP_K), 6))


In [ ]:
# CELL 9 — Refit on full data + infer for targets
full_user_features, full_item_features = build_user_item_features(interactions, editions, users, book_genres, interactions["event_ts"].max())
full_ug, full_ua = build_profiles(interactions, editions, book_genres)
full_als_bundle = fit_als(interactions)

test_cands = generate_candidates(targets["user_id"].tolist(), full_item_features, full_ug, full_ua, full_als_bundle)
test_nb = build_als_neighbor_features(test_cands, full_als_bundle, max_hist=10, knn=10)
test_cands = test_cands.join(test_nb, how="left").fillna(0)

# same table schema
test_rank = test_cands.merge(full_user_features, on="user_id", how="left")
test_rank = test_rank.merge(full_item_features, on="edition_id", how="left")

full_ug_user = full_ug.groupby("user_id")["user_genre_affinity"].agg(["mean","max"]).reset_index().rename(columns={"mean":"u_genre_aff_mean","max":"u_genre_aff_max"})
full_ua_user = full_ua.groupby("user_id")["user_author_affinity"].agg(["mean","max"]).reset_index().rename(columns={"mean":"u_author_aff_mean","max":"u_author_aff_max"})
test_rank = test_rank.merge(full_ug_user, on="user_id", how="left").merge(full_ua_user, on="user_id", how="left")
test_rank = test_rank.fillna(0)

for c in feature_cols:
    if c not in test_rank.columns:
        test_rank[c] = 0

pred = ranker.predict(Pool(test_rank[feature_cols]))
test_rank["pred"] = pred.astype("float32")


In [ ]:
# CELL 10 — Build submission + strict validation
submission = test_rank.sort_values(["user_id", "pred", "edition_id"], ascending=[True, False, True])
submission = submission.drop_duplicates(["user_id", "edition_id"], keep="first")
submission["rank"] = submission.groupby("user_id").cumcount() + 1
submission = submission[submission["rank"] <= TOP_K][["user_id", "edition_id", "rank"]].copy()

# fill missing users/ranks
global_top = full_item_features.sort_values("pop_log", ascending=False)["edition_id"].tolist()
seen_pairs = full_als_bundle["seen_pairs"]
out = []
for uid in targets["user_id"].tolist():
    g = submission[submission["user_id"] == uid].sort_values("rank")
    used = set()
    rank = 1
    for _, row in g.iterrows():
        eid = int(row["edition_id"])
        if eid not in used:
            out.append((int(uid), eid, rank))
            used.add(eid)
            rank += 1
    if rank <= TOP_K:
        for eid in global_top:
            eid = int(eid)
            if rank > TOP_K:
                break
            if eid in used or (uid, eid) in seen_pairs:
                continue
            out.append((int(uid), eid, rank))
            used.add(eid)
            rank += 1

submission_final = pd.DataFrame(out, columns=["user_id", "edition_id", "rank"])
submission_final = submission_final.sort_values(["user_id", "rank"]).reset_index(drop=True)
submission_final[["user_id", "edition_id", "rank"]] = submission_final[["user_id", "edition_id", "rank"]].astype("int64")

# strict checks
assert set(submission_final["user_id"].unique()) == set(targets["user_id"].unique()), "Missing target users"
assert submission_final.groupby("user_id").size().eq(TOP_K).all(), "Not exactly top-k per user"
assert not submission_final.duplicated(["user_id", "edition_id"]).any(), "Duplicated user-edition pairs"
assert submission_final.groupby("user_id")["rank"].apply(lambda s: set(s.tolist()) == set(range(1, TOP_K+1))).all(), "Bad rank sequence"

submission_final.to_csv("submission.csv", index=False)
print("submission.csv saved:", submission_final.shape)
submission_final.head(30)
